# Exploratory Notebook: Outlier Thresholds (exp_04)

**Goal:** Validate outlier detection thresholds through sensitivity analysis and define a severity-based flagging system for ablation studies.

**Outputs:**
- `outputs/phase_2/metadata/outlier_thresholds.json` (on Google Drive)
- Publication-quality plots saved to `outputs/phase_2/figures/exp_04_outlier_thresholds`


In [ ]:
# ============================================================
# RUNTIME SETTINGS
# ============================================================
# Required: CPU (Standard)
# GPU: Not required
# High-RAM: Recommended for large datasets
#
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets (for private repo access)
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from private GitHub repo
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("OK: Package installed")


In [ ]:
# Mount Google Drive for data files
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted")


In [ ]:
# Package imports
from urban_tree_transfer.config import PROJECT_CRS, RANDOM_SEED
from urban_tree_transfer.config.loader import (
    get_temporal_feature_names,
    load_feature_config,
)
from urban_tree_transfer.utils import ExecutionLog, save_figure, setup_plotting
from urban_tree_transfer.utils.plotting import PUBLICATION_STYLE

from pathlib import Path
import json

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, zscore, mannwhitneyu

try:
    from matplotlib_venn import venn3
except ImportError:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib-venn", "-q"], check=True)
    from matplotlib_venn import venn3

setup_plotting()
log = ExecutionLog("exp_04_outlier_thresholds")

print("OK: Package imports complete")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
INPUT_DIR = DRIVE_DIR / "data" / "phase_2_features"
OUTPUT_DIR = DRIVE_DIR / "data" / "phase_2_features"
METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "exp_04_outlier_thresholds"

CITIES = ["berlin", "leipzig"]
Z_THRESHOLDS = [2.5, 3.0, 3.5]
Z_MIN_FEATURE_COUNTS = [5, 10, 15]
MAHAL_ALPHA_LEVELS = [0.0001, 0.001, 0.01]
IQR_MULTIPLIERS = [1.5, 2.0, 3.0]

for d in [METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

feature_config = load_feature_config()

print(f"Input:  {INPUT_DIR}")
print(f"Output: {METADATA_DIR}")
print(f"Plots:  {FIGURES_DIR}")
print(f"Cities: {CITIES}")
print(f"Random seed: {RANDOM_SEED}")


## Data Loading & Validation

Load Phase 2b quality-controlled datasets and validate expected schema and CRS.


In [ ]:
log.start_step("Data Loading & Validation")

try:
    city_data = {}

    for city in CITIES:
        path = INPUT_DIR / f"trees_clean_{city}.gpkg"
        print(f"Loading: {path}")
        gdf = gpd.read_file(path)

        if gdf.crs is None or gdf.crs.to_epsg() != int(str(PROJECT_CRS).split(":")[-1]):
            raise ValueError(f"Invalid CRS for {city}: {gdf.crs}. Expected {PROJECT_CRS}.")

        gdf["genus_latin"] = gdf["genus_latin"].astype(str).str.upper().str.strip()
        gdf["city"] = city

        city_data[city] = gdf
        print(f"Loaded {city}: {len(gdf):,} rows, {len(gdf.columns):,} columns")

    log.end_step(status="success", records=sum(len(df) for df in city_data.values()))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Feature Column Selection

Use temporal Sentinel-2 features for Z-score and Mahalanobis detection, and CHM_1m for IQR.


In [ ]:
temporal_feature_cols = get_temporal_feature_names(config=feature_config)

# Only keep columns present in the dataset
available_cols = set(next(iter(city_data.values())).columns)
feature_cols = [c for c in temporal_feature_cols if c in available_cols]

missing_cols = [c for c in temporal_feature_cols if c not in available_cols]
if missing_cols:
    print(f"Warning: {len(missing_cols)} temporal feature columns missing and will be skipped.")

print(f"Using {len(feature_cols)} temporal feature columns for Z-score and Mahalanobis.")
print(f"CHM column present: {'CHM_1m' in available_cols}")


## Z-Score Sensitivity Analysis

Z-score threshold of 3.0 represents standard practice. Evaluate sensitivity across thresholds and minimum feature counts.


In [ ]:
log.start_step("Z-Score Sensitivity")

try:
    zscore_results = []
    zscore_flags = {}

    for city, gdf in city_data.items():
        features = gdf[feature_cols].astype(float)
        zscores = zscore(features, nan_policy="omit")
        zscores = pd.DataFrame(zscores, columns=feature_cols, index=gdf.index)

        for threshold in Z_THRESHOLDS:
            for min_count in Z_MIN_FEATURE_COUNTS:
                flagged = (zscores.abs() > threshold).sum(axis=1) >= min_count
                rate = flagged.mean()
                zscore_results.append({
                    "city": city,
                    "threshold": threshold,
                    "min_feature_count": min_count,
                    "flag_rate": rate,
                })

        # Cache standard choice for later use
        standard_flag = (zscores.abs() > 3.0).sum(axis=1) >= 10
        zscore_flags[city] = standard_flag

    zscore_df = pd.DataFrame(zscore_results)

    # Plot sensitivity
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, city in zip(axes[:2], CITIES):
        subset = zscore_df[zscore_df["city"] == city]
        for min_count in Z_MIN_FEATURE_COUNTS:
            line = subset[subset["min_feature_count"] == min_count]
            ax.plot(line["threshold"], line["flag_rate"], marker="o", label=f"min={min_count}")
        ax.set_title(f"Z-Score Sensitivity ({city})")
        ax.set_xlabel("Z-score threshold")
        ax.set_ylabel("Flagging rate")
        ax.legend()

    overall = zscore_df.groupby(["threshold", "min_feature_count"], as_index=False)["flag_rate"].mean()
    ax = axes[2]
    for min_count in Z_MIN_FEATURE_COUNTS:
        line = overall[overall["min_feature_count"] == min_count]
        ax.plot(line["threshold"], line["flag_rate"], marker="o", label=f"min={min_count}")
    ax.set_title("Z-Score Sensitivity (overall)")
    ax.set_xlabel("Z-score threshold")
    ax.set_ylabel("Flagging rate")
    ax.legend()

    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "zscore_sensitivity.png")
    plt.close(fig)

    log.end_step(status="success", records=len(zscore_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Mahalanobis alpha Validation

Mahalanobis alpha of 0.001 provides a conservative multivariate outlier threshold.
This section compares alpha values [0.0001, 0.001, 0.01] using chi^2 critical values and observed flagging rates.

Plots show the top genera by sample size for readability; alpha rates are computed across all genera.


In [ ]:
log.start_step("Mahalanobis Alpha Validation")

try:
    mahal_results = []
    mahal_flags = {}
    mahal_dist_rows = []
    
    # Track statistics
    skipped_genera = []
    processed_genera = []
    min_required = len(feature_cols) + 2
    
    print(f"Features: {len(feature_cols)}")
    print(f"Min samples per genus: {min_required}")
    print()

    for city, gdf in city_data.items():
        mahal_flags[city] = {}

        for genus, grp in gdf.groupby("genus_latin"):
            # Extract features and handle NaN values
            features = grp[feature_cols].astype(float)
            valid_mask = ~features.isna().any(axis=1)
            valid_features = features[valid_mask]
            
            # Skip genus if insufficient valid samples
            if len(valid_features) < min_required:
                skipped_genera.append((genus, city, len(valid_features)))
                continue
            
            processed_genera.append((genus, city, len(valid_features)))
            
            # Compute Mahalanobis distance
            mean_vec = valid_features.mean(axis=0).values
            cov = np.cov(valid_features.values, rowvar=False)
            
            # Use pseudo-inverse for numerical stability
            try:
                inv_cov = np.linalg.pinv(cov)
            except np.linalg.LinAlgError:
                print(f"  ⚠️ {city}/{genus}: Covariance matrix error - skipping")
                skipped_genera.append((genus, city, len(valid_features)))
                continue

            diff = valid_features.values - mean_vec
            d2 = np.einsum("ij,jk,ik->i", diff, inv_cov, diff)

            # Store distances for visualization
            for value in d2:
                mahal_dist_rows.append({
                    "city": city,
                    "genus_latin": genus,
                    "mahalanobis_d2": value,
                })

            # Test different alpha levels
            for alpha in MAHAL_ALPHA_LEVELS:
                threshold = chi2.ppf(1 - alpha, df=len(feature_cols))
                flagged = d2 > threshold

                mahal_results.append({
                    "city": city,
                    "genus_latin": genus,
                    "alpha": alpha,
                    "flag_rate": float(np.mean(flagged)),
                    "n_samples": len(valid_features),
                })

                # Cache standard choice (alpha=0.001) for consensus
                if alpha == 0.001:
                    # Map flags back to original indices (including NaN rows as False)
                    genus_flags = pd.Series(False, index=grp.index)
                    genus_flags.loc[grp.index[valid_mask]] = flagged
                    mahal_flags[city][genus] = genus_flags

    # Print summary
    print(f"{'='*60}")
    print(f"Processed genera: {len(set(g[0] for g in processed_genera))}")
    print(f"Skipped genera:   {len(set(g[0] for g in skipped_genera))}")
    
    if skipped_genera:
        print(f"\nSkipped genera (< {min_required} valid samples):")
        for genus, city, n_samples in sorted(set(skipped_genera), key=lambda x: (x[0], x[1])):
            print(f"  {genus:20s} ({city:7s}): {n_samples:4d} samples")
    print(f"{'='*60}\n")
    
    # Handle empty results
    if not mahal_results:
        raise RuntimeError(
            f"No genera had sufficient samples for Mahalanobis analysis.\n"
            f"Required: {min_required} samples per genus.\n"
            f"Check data quality - all genera were skipped."
        )
    
    # Convert to DataFrames
    mahal_df = pd.DataFrame(mahal_results)
    mahal_dist_df = pd.DataFrame(mahal_dist_rows)

    # Compute weighted average flagging rates
    alpha_rates_list = []
    
    # Per-city rates
    for city in CITIES:
        city_subset = mahal_df[mahal_df["city"] == city]
        if city_subset.empty:
            continue
        for alpha in MAHAL_ALPHA_LEVELS:
            alpha_subset = city_subset[city_subset["alpha"] == alpha]
            if alpha_subset.empty or alpha_subset["n_samples"].sum() == 0:
                continue
            weighted_rate = np.average(
                alpha_subset["flag_rate"], 
                weights=alpha_subset["n_samples"]
            )
            alpha_rates_list.append({
                "city": city,
                "alpha": alpha,
                "flag_rate": weighted_rate
            })
    
    # Overall rates
    for alpha in MAHAL_ALPHA_LEVELS:
        alpha_subset = mahal_df[mahal_df["alpha"] == alpha]
        if alpha_subset.empty or alpha_subset["n_samples"].sum() == 0:
            continue
        weighted_rate = np.average(
            alpha_subset["flag_rate"],
            weights=alpha_subset["n_samples"]
        )
        alpha_rates_list.append({
            "city": "overall",
            "alpha": alpha,
            "flag_rate": weighted_rate
        })
    
    alpha_rates = pd.DataFrame(alpha_rates_list)

    # Plot D^2 distributions (top 8 genera by sample size)
    top_genera = (
        mahal_dist_df["genus_latin"].value_counts().head(8).index.tolist()
    )
    plot_df = mahal_dist_df[mahal_dist_df["genus_latin"].isin(top_genera)]

    g = sns.FacetGrid(plot_df, col="genus_latin", col_wrap=4, height=2.5, sharex=True, sharey=True)
    g.map_dataframe(sns.histplot, x="mahalanobis_d2", bins=40, color="#4c72b0")

    for ax in g.axes.flatten():
        for alpha in MAHAL_ALPHA_LEVELS:
            threshold = chi2.ppf(1 - alpha, df=len(feature_cols))
            ax.axvline(threshold, linestyle="--", linewidth=1, label=f"alpha={alpha}")
        ax.set_xlabel("D^2")
        ax.set_ylabel("Count")

    # Add inset with overall flagging rates
    inset_ax = g.fig.add_axes([0.72, 0.05, 0.26, 0.30])
    overall_subset = alpha_rates[alpha_rates["city"] == "overall"]
    if not overall_subset.empty:
        inset_ax.bar(
            overall_subset["alpha"].astype(str), 
            overall_subset["flag_rate"], 
            color="#55a868"
        )
        inset_ax.set_title("Overall flagging rate")
        inset_ax.set_xlabel("alpha")
        inset_ax.set_ylabel("Rate")
        inset_ax.tick_params(axis="x", labelrotation=45)

    g.fig.suptitle("Mahalanobis D^2 Distributions by Genus with alpha Thresholds", y=1.02)
    handles, labels = g.axes.flatten()[0].get_legend_handles_labels()
    g.fig.legend(handles, labels, title="Chi^2 thresholds", loc="upper center", ncol=3)
    plt.tight_layout()
    save_figure(g.fig, FIGURES_DIR / "mahalanobis_distribution.png")
    plt.close(g.fig)

    log.end_step(status="success", records=len(mahal_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## IQR Multiplier Evaluation

IQR multiplier of 1.5 is standard for box plot outlier detection.


In [ ]:
log.start_step("IQR Multiplier Evaluation")

try:
    iqr_results = []
    iqr_flags = {}

    for city, gdf in city_data.items():
        iqr_flags[city] = {}
        for (genus, grp) in gdf.groupby(["genus_latin"]):
            values = grp["CHM_1m"].astype(float)

            for k in IQR_MULTIPLIERS:
                q1 = values.quantile(0.25)
                q3 = values.quantile(0.75)
                iqr = q3 - q1
                lower = q1 - k * iqr
                upper = q3 + k * iqr
                flagged = (values < lower) | (values > upper)

                iqr_results.append({
                    "city": city,
                    "genus_latin": genus,
                    "multiplier": k,
                    "flag_rate": float(flagged.mean()),
                })

                if k == 1.5:
                    iqr_flags[city][genus] = pd.Series(flagged, index=grp.index)

    iqr_df = pd.DataFrame(iqr_results)

    plot_df = pd.concat(city_data.values(), ignore_index=True)
    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    sns.boxplot(
        data=plot_df,
        x="genus_latin",
        y="CHM_1m",
        hue="city",
        showfliers=False,
        ax=ax,
    )
    ax.set_title("CHM_1m Distribution by Genus and City (Tukey Fences)")
    ax.set_xlabel("Genus")
    ax.set_ylabel("CHM_1m")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "iqr_boxplots_per_genus.png")
    plt.close(fig)

    log.end_step(status="success", records=len(iqr_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Severity-Based Flagging System

Combine Z-score, Mahalanobis, and IQR flags into severity categories.
No trees are removed; all trees proceed to Phase 2c with flags attached.


In [ ]:
log.start_step("Severity Flagging")

try:
    combined = []

    for city, gdf in city_data.items():
        z_flag = zscore_flags[city]

        m_flag = pd.Series(False, index=gdf.index)
        for genus, series in mahal_flags[city].items():
            m_flag.loc[series.index] = series.values

        i_flag = pd.Series(False, index=gdf.index)
        for genus, series in iqr_flags[city].items():
            i_flag.loc[series.index] = series.values

        severity_count = z_flag.astype(int) + m_flag.astype(int) + i_flag.astype(int)
        severity_label = pd.Series("none", index=gdf.index)
        severity_label[severity_count == 1] = "low"
        severity_label[severity_count == 2] = "medium"
        severity_label[severity_count == 3] = "high"

        result = gdf[["tree_id", "city", "genus_latin"]].copy()
        result["outlier_zscore"] = z_flag.values
        result["outlier_mahalanobis"] = m_flag.values
        result["outlier_iqr"] = i_flag.values
        result["outlier_severity"] = severity_label.values
        combined.append(result)

    flagged_df = pd.concat(combined, ignore_index=True)

    log.end_step(status="success", records=len(flagged_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Outlier Verification & Visual Inspection

Verify that identified outliers are truly anomalous data points vs. biologically extreme but valid trees (e.g., large mature oaks, old platanus).

**Visualizations:**
- PCA feature space plot (outliers should be separated if truly anomalous)
- Feature distribution comparisons (CHM, NDVI, spectral bands)
- Representative examples table (top 5 genera)
- Interpretation guide for biological vs. data quality outliers

In [ ]:
log.start_step("Outlier Verification & Visualization")

try:
    # Combine data for visualization
    combined_gdf = pd.concat(city_data.values(), ignore_index=True)
    combined_flags = pd.concat(combined, ignore_index=True)
    
    viz_data = combined_gdf.merge(
        combined_flags[["tree_id", "outlier_severity", "outlier_zscore", 
                        "outlier_mahalanobis", "outlier_iqr"]], 
        on="tree_id", 
        how="left"
    )
    
    severity_colors = {
        "none": "#2ca02c",
        "low": "#ffbb78",
        "medium": "#ff7f0e",
        "high": "#d62728"
    }
    
    # ========================================
    # 1. PCA Feature Space Visualization
    # ========================================
    print("Creating PCA visualization...")
    
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    
    pca_features = feature_cols[:50] + ["CHM_1m"]
    pca_data = viz_data[pca_features].fillna(viz_data[pca_features].median())
    
    scaler = StandardScaler()
    pca_scaled = scaler.fit_transform(pca_data)
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(pca_scaled)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: All severities
    ax = axes[0]
    for severity in ["none", "low", "medium", "high"]:
        mask = viz_data["outlier_severity"] == severity
        ax.scatter(
            pca_result[mask, 0], 
            pca_result[mask, 1],
            c=severity_colors[severity],
            label=f"{severity} (n={mask.sum():,})",
            alpha=0.6 if severity == "none" else 0.9,
            s=10 if severity == "none" else 30,
            edgecolors='black' if severity == "high" else 'none',
            linewidths=0.5
        )
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("PCA: Feature Space Distribution by Outlier Severity")
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Plot 2: High-severity vs. none
    ax = axes[1]
    for severity in ["none", "high"]:
        mask = viz_data["outlier_severity"] == severity
        ax.scatter(
            pca_result[mask, 0], 
            pca_result[mask, 1],
            c=severity_colors[severity],
            label=f"{severity} (n={mask.sum():,})",
            alpha=0.4 if severity == "none" else 1.0,
            s=5 if severity == "none" else 50,
            edgecolors='black' if severity == "high" else 'none',
            linewidths=0.8,
            marker='*' if severity == "high" else 'o'
        )
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("PCA: High-Severity Outliers vs. Normal Trees")
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "outlier_pca_verification.png", dpi=300)
    plt.close(fig)
    
    # ========================================
    # 2. Feature Distribution Comparisons
    # ========================================
    print("Creating feature distribution comparisons...")
    
    key_features = ["CHM_1m", "NDVI_06", "B8_06", "EVI_06", "NDWI_06"]
    key_features = [f for f in key_features if f in viz_data.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(key_features):
        ax = axes[idx]
        
        data_by_severity = [
            viz_data[viz_data["outlier_severity"] == sev][feature].dropna()
            for sev in ["none", "low", "medium", "high"]
        ]
        
        bp = ax.boxplot(
            data_by_severity,
            labels=["none", "low", "medium", "high"],
            patch_artist=True,
            showfliers=False
        )
        
        for patch, severity in zip(bp['boxes'], ["none", "low", "medium", "high"]):
            patch.set_facecolor(severity_colors[severity])
            patch.set_alpha(0.7)
        
        high_data = viz_data[viz_data["outlier_severity"] == "high"][feature].dropna()
        if len(high_data) > 0:
            ax.scatter(
                [4] * len(high_data),
                high_data,
                color=severity_colors["high"],
                alpha=0.6,
                s=20,
                zorder=3
            )
        
        ax.set_title(f"{feature}")
        ax.set_xlabel("Outlier Severity")
        ax.set_ylabel("Value")
        ax.grid(axis='y', alpha=0.3)
    
    if len(key_features) < 6:
        fig.delaxes(axes[-1])
    
    plt.suptitle("Feature Distributions by Outlier Severity\n(High-severity points shown individually)", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "outlier_feature_distributions.png", dpi=300)
    plt.close(fig)
    
    # ========================================
    # 3. Representative Examples Table
    # ========================================
    print("Extracting representative examples...")
    
    high_severity = viz_data[viz_data["outlier_severity"] == "high"].copy()
    
    if len(high_severity) > 0:
        examples = []
        for genus in high_severity["genus_latin"].unique()[:5]:
            genus_subset = high_severity[high_severity["genus_latin"] == genus]
            if len(genus_subset) > 0:
                example = genus_subset.nlargest(1, "CHM_1m").iloc[0]
                examples.append({
                    "tree_id": example["tree_id"],
                    "city": example["city"],
                    "genus": example["genus_latin"],
                    "CHM_1m": f"{example['CHM_1m']:.1f}m",
                    "NDVI_06": f"{example.get('NDVI_06', np.nan):.3f}",
                    "plant_year": int(example["plant_year"]) if pd.notna(example["plant_year"]) else "N/A",
                    "z_flag": "✓" if example["outlier_zscore"] else "✗",
                    "m_flag": "✓" if example["outlier_mahalanobis"] else "✗",
                    "iqr_flag": "✓" if example["outlier_iqr"] else "✗",
                })
        
        examples_df = pd.DataFrame(examples)
        
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.axis('tight')
        ax.axis('off')
        
        table = ax.table(
            cellText=examples_df.values,
            colLabels=examples_df.columns,
            cellLoc='center',
            loc='center',
            colWidths=[0.12, 0.08, 0.15, 0.10, 0.10, 0.12, 0.08, 0.08, 0.08]
        )
        
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2)
        
        for i in range(len(examples_df.columns)):
            table[(0, i)].set_facecolor('#4472C4')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        for i in range(1, len(examples_df) + 1):
            for j in range(len(examples_df.columns)):
                table[(i, j)].set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
        
        plt.title("Representative High-Severity Outliers\n(Most extreme examples per genus)", 
                  fontsize=12, fontweight='bold', pad=20)
        plt.tight_layout()
        save_figure(fig, FIGURES_DIR / "outlier_examples_table.png", dpi=300)
        plt.close(fig)
        
        print("\nHigh-Severity Outlier Examples:")
        print(examples_df.to_string(index=False))
    
    # ========================================
    # 4. Summary Statistics
    # ========================================
    print("\n" + "="*60)
    print("OUTLIER VERIFICATION SUMMARY")
    print("="*60)
    
    for severity in ["high", "medium", "low"]:
        subset = viz_data[viz_data["outlier_severity"] == severity]
        if len(subset) == 0:
            continue
        
        print(f"\n{severity.upper()} Severity (n={len(subset):,}):")
        print(f"  CHM_1m:    μ={subset['CHM_1m'].mean():.1f}m, σ={subset['CHM_1m'].std():.1f}m")
        if "NDVI_06" in subset.columns:
            print(f"  NDVI_06:   μ={subset['NDVI_06'].mean():.3f}, σ={subset['NDVI_06'].std():.3f}")
        if "plant_year" in subset.columns:
            valid_years = subset["plant_year"].dropna()
            if len(valid_years) > 0:
                print(f"  Plant Year: μ={valid_years.mean():.0f}, median={valid_years.median():.0f}")
        
        genus_dist = subset["genus_latin"].value_counts().head(3)
        print(f"  Top Genera: {', '.join(f'{g}({c})' for g, c in genus_dist.items())}")
    
    print("\n" + "="*60)
    print("INTERPRETATION GUIDE:")
    print("="*60)
    print("✓ Outliers in PCA space clearly separated → likely data quality issues")
    print("✓ Outliers mixed with normal trees → likely biological extremes")
    print("✓ High CHM + old plant_year → large mature trees (valid)")
    print("✓ Unusual NDVI patterns → potential sensor issues or dead trees")
    print("="*60)
    
    log.end_step(status="success", records=len(viz_data))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Section 5: Biological Context Analysis

Explore whether high-severity outliers are associated with biological characteristics that could explain extreme values (age, tree type, location).

**Purpose:** Distinguish data quality issues from biological extremes.

**Analyses:**
1. Age distribution (plant_year correlation)
2. Tree type association (anlagenbaeume vs. strassenbaeume)
3. Spatial clustering (Ripley's K test)
4. Genus-specific outlier rates

**Note:** This is exploratory analysis. Outlier flags remain unchanged - interpretation only.


In [ ]:
# Prepare Berlin context data for biological analyses
trees_berlin = city_data.get("berlin")
if trees_berlin is None:
    raise RuntimeError("Berlin data not loaded - required for biological context analysis")

berlin_flags = flagged_df[flagged_df["city"] == "berlin"][
    ["tree_id", "outlier_severity"]
]
trees_berlin = trees_berlin.merge(berlin_flags, on="tree_id", how="left")

age_stats = {"interpretation": "not evaluated"}
tree_type_stats = {"interpretation": "not evaluated"}
spatial_stats = {"interpretation": "not evaluated"}
genus_stats = {"interpretation": "not evaluated"}


In [ ]:
log.start_step("Biological Context Analysis - Age Distribution")

try:
    trees_with_age = trees_berlin[trees_berlin["plant_year"].notna()].copy()

    if len(trees_with_age) < 100:
        print("⚠️ Insufficient plant_year data - skipping age analysis")
        age_stats = {"interpretation": "Insufficient plant_year data"}
        log.end_step(status="skipped")
    else:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        severities = ["high", "medium", "low", "none"]
        colors = {"high": "#d62728", "medium": "#ff7f0e", "low": "#ffbb78", "none": "#2ca02c"}

        for idx, severity in enumerate(severities):
            ax = axes[idx]
            subset = trees_with_age[trees_with_age["outlier_severity"] == severity]
            none_subset = trees_with_age[trees_with_age["outlier_severity"] == "none"]

            if len(subset) > 0:
                ax.hist(
                    subset["plant_year"].dropna(),
                    bins=30,
                    alpha=0.7,
                    color=colors[severity],
                    label=f"{severity} (n={len(subset)})",
                )
                ax.hist(
                    none_subset["plant_year"].dropna(),
                    bins=30,
                    alpha=0.4,
                    color="gray",
                    label=f"none (n={len(none_subset)})",
                )

                median_val = subset["plant_year"].median()
                ax.axvline(median_val, color=colors[severity], linestyle="--", linewidth=2)
                ax.text(
                    median_val,
                    ax.get_ylim()[1] * 0.9,
                    f"Median: {int(median_val)}",
                    rotation=90,
                    va="top",
                )

            ax.set_title(f"Severity: {severity.upper()}", fontsize=12)
            ax.set_xlabel("Plant Year", fontsize=10)
            ax.set_ylabel("Count", fontsize=10)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)

        plt.suptitle("Plant Year Distribution by Outlier Severity", fontsize=14, fontweight="bold")
        plt.tight_layout()
        save_figure(fig, FIGURES_DIR / "outlier_age_distribution.png", dpi=300)
        plt.close(fig)
        print(f"Saved: {FIGURES_DIR / 'outlier_age_distribution.png'}")

        high_severity = trees_with_age[trees_with_age["outlier_severity"] == "high"]
        no_outlier = trees_with_age[trees_with_age["outlier_severity"] == "none"]

        if len(high_severity) > 30 and len(no_outlier) > 30:
            u_stat, p_value = mannwhitneyu(
                high_severity["plant_year"].dropna(),
                no_outlier["plant_year"].dropna(),
                alternative="less",
            )

            print("\nMann-Whitney U Test (High vs. None):")
            print(f"  U-statistic: {u_stat}")
            print(f"  p-value: {p_value:.4f}")

            if p_value < 0.05:
                print("  ✅ High-severity outliers are significantly OLDER (p < 0.05)")
                interpretation = (
                    "High-severity outliers significantly older - likely biological extremes"
                )
            else:
                print("  ❌ No significant age difference (p >= 0.05)")
                interpretation = "No significant age correlation - likely data quality issues"

            age_stats = {
                "high_severity_median_plant_year": int(high_severity["plant_year"].median()),
                "non_outlier_median_plant_year": int(no_outlier["plant_year"].median()),
                "mann_whitney_u": float(u_stat),
                "p_value": float(p_value),
                "interpretation": interpretation,
            }
        else:
            age_stats = {
                "interpretation": "Insufficient data for statistical test"
            }

        log.end_step(status="success")

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


In [ ]:
log.start_step("Biological Context Analysis - Tree Type")

try:
    if "tree_type" not in trees_berlin.columns:
        print("⚠️ tree_type column not found - skipping tree type analysis")
        tree_type_stats = {"interpretation": "tree_type data not available"}
        log.end_step(status="skipped")
    else:
        contingency = pd.crosstab(
            trees_berlin["outlier_severity"],
            trees_berlin["tree_type"],
            normalize="index",
        )

        print("\nTree Type Distribution by Outlier Severity:")
        print(contingency.round(3))

        fig, ax = plt.subplots(figsize=(10, 6))
        contingency.plot(kind="bar", stacked=True, ax=ax, color=["#8dd3c7", "#fb8072"])

        ax.set_title("Tree Type Distribution by Outlier Severity", fontsize=14)
        ax.set_xlabel("Outlier Severity", fontsize=12)
        ax.set_ylabel("Proportion", fontsize=12)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.legend(title="Tree Type", fontsize=10)
        ax.grid(True, axis="y", alpha=0.3)

        plt.tight_layout()
        save_figure(fig, FIGURES_DIR / "outlier_tree_type.png", dpi=300)
        plt.close(fig)
        print(f"Saved: {FIGURES_DIR / 'outlier_tree_type.png'}")

        if "anlagenbaeume" in contingency.columns and "high" in contingency.index:
            high_anlagen_pct = contingency.loc["high", "anlagenbaeume"]
            none_anlagen_pct = contingency.loc["none", "anlagenbaeume"] if "none" in contingency.index else 0

            tree_type_stats = {
                "high_severity_anlagenbaeume_pct": float(high_anlagen_pct),
                "non_outlier_anlagenbaeume_pct": float(none_anlagen_pct),
                "interpretation": (
                    "High-severity outliers enriched in park trees (anlagenbaeume)"
                    if high_anlagen_pct > none_anlagen_pct + 0.1
                    else "No strong tree type association"
                ),
            }
        else:
            tree_type_stats = {"interpretation": "Insufficient tree type categories"}

        log.end_step(status="success")

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


In [ ]:
log.start_step("Biological Context Analysis - Spatial Clustering")

try:
    from pointpats import ripley

    high_severity_trees = trees_berlin[trees_berlin["outlier_severity"] == "high"].copy()

    if len(high_severity_trees) < 30:
        print("⚠️ Too few high-severity outliers for spatial analysis")
        spatial_stats = {"interpretation": "Insufficient high-severity outliers"}
        log.end_step(status="skipped")
    else:
        coords = np.column_stack([
            high_severity_trees.geometry.x,
            high_severity_trees.geometry.y,
        ])

        distances = [50, 100, 200]
        k_results = {}

        for d in distances:
            k_observed = ripley.k(coords, support=d)
            k_expected = np.pi * d ** 2

            k_results[f"{d}m"] = {
                "observed": float(k_observed[-1]) if len(k_observed) > 0 else np.nan,
                "expected": float(k_expected),
                "ratio": (
                    float(k_observed[-1] / k_expected)
                    if len(k_observed) > 0 and k_expected > 0
                    else np.nan
                ),
            }

            print(f"Ripley's K at {d}m:")
            print(f"  Observed: {k_results[f'{d}m']['observed']:.1f}")
            print(f"  Expected (CSR): {k_results[f'{d}m']['expected']:.1f}")
            print(f"  Ratio: {k_results[f'{d}m']['ratio']:.2f}")

        if k_results["100m"]["ratio"] > 1.2:
            interpretation = "High-severity outliers spatially clustered (likely parks/monuments)"
        else:
            interpretation = "High-severity outliers spatially random (not clustered)"

        spatial_stats = {
            "ripleys_k": k_results,
            "interpretation": interpretation,
        }


        # Visualization: Spatial clustering map
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Plot all trees in background
        trees_berlin.plot(ax=ax, color="lightgray", markersize=1, alpha=0.3, label="Non-outliers")
        
        # Overlay high-severity outliers
        high_severity_trees.plot(ax=ax, color="red", markersize=20, alpha=0.7, 
                                 marker="*", label="High-severity outliers")
        
        # Add Ripley's K results as text annotation
        k_text = "Ripley's K (spatial clustering):\n"
        for d, stats in k_results.items():
            ratio = stats["ratio"]
            k_text += f"  {d}: K_ratio = {ratio:.2f}"
            if ratio > 1.2:
                k_text += " (clustered) ✓\n"
            else:
                k_text += "\n"
        
        ax.text(0.02, 0.98, k_text, transform=ax.transAxes, 
                fontsize=10, verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
        
        ax.set_title("Spatial Distribution: High-Severity Outliers vs. All Trees\n" + 
                     "(Clustering indicates biological contexts like parks)", 
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("X (m)", fontsize=11)
        ax.set_ylabel("Y (m)", fontsize=11)
        ax.legend(loc="upper right", fontsize=10)
        ax.grid(alpha=0.3)
        
        save_figure(fig, FIGURES_DIR / "outlier_spatial_clustering.png", dpi=dpi)
        
        log.end_step(status="success")

except ImportError:
    print("⚠️ pointpats library not available - skipping Ripley's K analysis")
    spatial_stats = {"interpretation": "pointpats library not installed"}
    log.end_step(status="skipped")
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


In [ ]:
log.start_step("Biological Context Analysis - Genus Patterns")

try:
    genus_outlier_rates = (
        trees_berlin.groupby("genus_latin")["outlier_severity"]
        .apply(lambda x: (x == "high").mean())
        .sort_values(ascending=False)
    )

    genus_counts = trees_berlin["genus_latin"].value_counts()
    viable_genera = genus_counts[genus_counts >= 500].index
    genus_outlier_rates = genus_outlier_rates[genus_outlier_rates.index.isin(viable_genera)]

    print("\nHigh-Severity Outlier Rates by Genus (top 10):")
    print(genus_outlier_rates.head(10).round(3))

    fig, ax = plt.subplots(figsize=(12, 6))
    genus_outlier_rates.head(10).plot(kind="bar", ax=ax, color="steelblue")

    ax.set_title("High-Severity Outlier Rate by Genus", fontsize=14)
    ax.set_xlabel("Genus", fontsize=12)
    ax.set_ylabel("High-Severity Rate", fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "outlier_genus_rates.png", dpi=300)
    plt.close(fig)
    print(f"Saved: {FIGURES_DIR / 'outlier_genus_rates.png'}")

    genus_stats = {
        "top_outlier_genera": genus_outlier_rates.head(5).to_dict(),
        "mean_rate": float(genus_outlier_rates.mean()),
        "std_rate": float(genus_outlier_rates.std()),
        "interpretation": (
            "QUERCUS, PLATANUS show higher rates (large trees with high variability)"
            if "QUERCUS" in genus_outlier_rates.head(3).index
            else "Outlier rates relatively uniform across genera"
        ),
    }

    log.end_step(status="success")

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Method Overlap Analysis and Summary Statistics


In [ ]:
log.start_step("Method Overlap Analysis")

try:
    z = flagged_df["outlier_zscore"]
    m = flagged_df["outlier_mahalanobis"]
    i = flagged_df["outlier_iqr"]

    overlap_counts = {
        "zscore_only": int((z & ~m & ~i).sum()),
        "mahalanobis_only": int((~z & m & ~i).sum()),
        "iqr_only": int((~z & ~m & i).sum()),
        "zscore_and_mahalanobis": int((z & m & ~i).sum()),
        "zscore_and_iqr": int((z & ~m & i).sum()),
        "mahalanobis_and_iqr": int((~z & m & i).sum()),
        "all_three": int((z & m & i).sum()),
    }

    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    venn3(
        subsets=(
            overlap_counts["zscore_only"],
            overlap_counts["mahalanobis_only"],
            overlap_counts["zscore_and_mahalanobis"],
            overlap_counts["iqr_only"],
            overlap_counts["zscore_and_iqr"],
            overlap_counts["mahalanobis_and_iqr"],
            overlap_counts["all_three"],
        ),
        set_labels=("Z-score", "Mahalanobis", "IQR"),
        ax=ax,
    )
    ax.set_title("Outlier Method Overlap (All Cities)")
    save_figure(fig, FIGURES_DIR / "outlier_venn_diagram.png")
    plt.close(fig)

    log.end_step(status="success", records=len(flagged_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Consensus Decision Flow

Visual flow from all trees to 1/2/3-method flags and final severity categories.


In [ ]:
log.start_step("Consensus Decision Flow")

try:
    total = len(flagged_df)
    flag_sum = flagged_df[["outlier_zscore", "outlier_mahalanobis", "outlier_iqr"]].sum(axis=1)
    count_1 = int((flag_sum == 1).sum())
    count_2 = int((flag_sum == 2).sum())
    count_3 = int((flag_sum == 3).sum())
    count_0 = int((flag_sum == 0).sum())

    severity_counts = flagged_df["outlier_severity"].value_counts()
    sev_none = int(severity_counts.get("none", 0))
    sev_low = int(severity_counts.get("low", 0))
    sev_medium = int(severity_counts.get("medium", 0))
    sev_high = int(severity_counts.get("high", 0))

    def pct(x):
        return 100 * x / total if total else 0

    import matplotlib.patches as patches

    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    ax.axis("off")

    # Node positions (x, y)
    nodes = {
        "all": (0.05, 0.5, f"All trees\n{total:,} (100%)"),
        "f0": (0.35, 0.8, f"0 methods\n{count_0:,} ({pct(count_0):.1f}%)"),
        "f1": (0.35, 0.55, f"1 method\n{count_1:,} ({pct(count_1):.1f}%)"),
        "f2": (0.35, 0.30, f"2 methods\n{count_2:,} ({pct(count_2):.1f}%)"),
        "f3": (0.35, 0.05, f"3 methods\n{count_3:,} ({pct(count_3):.1f}%)"),
        "none": (0.75, 0.8, f"None\n{sev_none:,} ({pct(sev_none):.1f}%)"),
        "low": (0.75, 0.55, f"Low\n{sev_low:,} ({pct(sev_low):.1f}%)"),
        "medium": (0.75, 0.30, f"Medium\n{sev_medium:,} ({pct(sev_medium):.1f}%)"),
        "high": (0.75, 0.05, f"High\n{sev_high:,} ({pct(sev_high):.1f}%)"),
    }

    def draw_node(x, y, text, color):
        box = patches.FancyBboxPatch(
            (x, y), 0.18, 0.12,
            boxstyle="round,pad=0.02",
            linewidth=1, edgecolor="#333333", facecolor=color
        )
        ax.add_patch(box)
        ax.text(x + 0.09, y + 0.06, text, ha="center", va="center", fontsize=10)

    draw_node(*nodes["all"][:2], nodes["all"][2], "#d9edf7")
    draw_node(*nodes["f0"][:2], nodes["f0"][2], "#f0f0f0")
    draw_node(*nodes["f1"][:2], nodes["f1"][2], "#fff2cc")
    draw_node(*nodes["f2"][:2], nodes["f2"][2], "#ffe599")
    draw_node(*nodes["f3"][:2], nodes["f3"][2], "#f4cccc")
    draw_node(*nodes["none"][:2], nodes["none"][2], "#f0f0f0")
    draw_node(*nodes["low"][:2], nodes["low"][2], "#fff2cc")
    draw_node(*nodes["medium"][:2], nodes["medium"][2], "#ffe599")
    draw_node(*nodes["high"][:2], nodes["high"][2], "#f4cccc")

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", lw=1, color="#555555"))

    # Left -> middle
    for key in ["f0", "f1", "f2", "f3"]:
        arrow(0.23, 0.56, nodes[key][0], nodes[key][1] + 0.06)

    # Middle -> right (map to severity)
    arrow(0.53, 0.86, nodes["none"][0], nodes["none"][1] + 0.06)
    arrow(0.53, 0.61, nodes["low"][0], nodes["low"][1] + 0.06)
    arrow(0.53, 0.36, nodes["medium"][0], nodes["medium"][1] + 0.06)
    arrow(0.53, 0.11, nodes["high"][0], nodes["high"][1] + 0.06)

    ax.set_title("Consensus Decision Flow")
    save_figure(fig, FIGURES_DIR / "consensus_decision_flow.png")
    plt.close(fig)

    log.end_step(status="success", records=total)

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Outlier Rates, Severity Distribution, and Genus Impact


In [ ]:
log.start_step("Outlier Rates and Genus Impact")

try:
    rate_rows = []
    for city in CITIES:
        subset = flagged_df[flagged_df["city"] == city]
        rate_rows.append({
            "city": city,
            "method": "zscore",
            "rate": subset["outlier_zscore"].mean(),
        })
        rate_rows.append({
            "city": city,
            "method": "mahalanobis",
            "rate": subset["outlier_mahalanobis"].mean(),
        })
        rate_rows.append({
            "city": city,
            "method": "iqr",
            "rate": subset["outlier_iqr"].mean(),
        })

    overall = {
        "zscore": flagged_df["outlier_zscore"].mean(),
        "mahalanobis": flagged_df["outlier_mahalanobis"].mean(),
        "iqr": flagged_df["outlier_iqr"].mean(),
    }
    for method, rate in overall.items():
        rate_rows.append({
            "city": "overall",
            "method": method,
            "rate": rate,
        })

    rates_df = pd.DataFrame(rate_rows)

    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    sns.barplot(data=rates_df, x="method", y="rate", hue="city", ax=ax)
    ax.set_title("Outlier Rates per Method")
    ax.set_xlabel("Method")
    ax.set_ylabel("Flagging rate")
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "outlier_rates_per_method.png")
    plt.close(fig)

    severity_counts = (
        flagged_df.groupby(["city", "outlier_severity"])
        .size()
        .reset_index(name="count")
    )
    overall_counts = (
        flagged_df.groupby("outlier_severity")
        .size()
        .reset_index(name="count")
    )
    overall_counts["city"] = "overall"
    severity_counts = pd.concat([severity_counts, overall_counts], ignore_index=True)

    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    sns.barplot(data=severity_counts, x="outlier_severity", y="count", hue="city", ax=ax)
    ax.set_title("Outlier Severity Distribution")
    ax.set_xlabel("Severity")
    ax.set_ylabel("Count")
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "severity_distribution.png")
    plt.close(fig)

    genus_sev = (
        flagged_df.groupby(["genus_latin", "outlier_severity"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    bottom = np.zeros(len(genus_sev))
    for severity in ["none", "low", "medium", "high"]:
        counts = genus_sev.get(severity, pd.Series(0, index=genus_sev.index))
        ax.bar(genus_sev.index, counts, bottom=bottom, label=severity)
        bottom += counts.values
    ax.set_title("Outlier Severity by Genus")
    ax.set_xlabel("Genus")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Severity")
    plt.tight_layout()
    save_figure(fig, FIGURES_DIR / "outlier_by_genus.png")
    plt.close(fig)

    log.end_step(status="success", records=len(flagged_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Validation Checks

- Severity distribution should be dominated by "none"
- No genus should have >2x the average flagging rate
- Majority of flagged trees should be flagged by only one method


In [ ]:
severity_rates = flagged_df["outlier_severity"].value_counts(normalize=True)
print("Severity rates:")
print(severity_rates)

if severity_rates.get("none", 0.0) < 0.8:
    print("Warning: 'none' share below 80% -- check thresholds.")

flag_rate_by_genus = (
    flagged_df.assign(any_flag=flagged_df[["outlier_zscore", "outlier_mahalanobis", "outlier_iqr"]].any(axis=1))
    .groupby("genus_latin")["any_flag"]
    .mean()
)
avg_rate = flag_rate_by_genus.mean()
high_genera = flag_rate_by_genus[flag_rate_by_genus > 2 * avg_rate]
if not high_genera.empty:
    print("Warning: Genera with >2x average flagging rate:")
    print(high_genera.sort_values(ascending=False))
    print("Genus names: " + ", ".join(high_genera.sort_values(ascending=False).index.tolist()))
else:
    print("Genus-wise flagging rates are within 2x average.")

flag_sum = flagged_df[["outlier_zscore", "outlier_mahalanobis", "outlier_iqr"]].sum(axis=1)
overlap_distribution = flag_sum.value_counts(normalize=True)
print("Overlap distribution (0/1/2/3 methods):")
print(overlap_distribution.sort_index())

if overlap_distribution.get(1, 0.0) < overlap_distribution.get(2, 0.0):
    print("Warning: More 2-method flags than 1-method flags -- check overlap.")

summary_table = (
    flagged_df.groupby(["city", "outlier_severity"]).size()
    .unstack(fill_value=0)
    .assign(total=lambda df: df.sum(axis=1))
)
print("\nSeverity counts by city:")
print(summary_table)


## Export Configuration JSON

Store final thresholds and statistics for Phase 2c usage.


In [ ]:
per_city_stats = {}
for city in CITIES:
    subset = flagged_df[flagged_df["city"] == city]
    counts = subset["outlier_severity"].value_counts()
    per_city_stats[city] = {
        "total_trees": int(len(subset)),
        "high": int(counts.get("high", 0)),
        "medium": int(counts.get("medium", 0)),
        "low": int(counts.get("low", 0)),
    }

severity_rates = flagged_df["outlier_severity"].value_counts(normalize=True)

output_json = {
    "zscore": {
        "threshold": 3.0,
        "min_feature_count": 10,
        "justification": "Standard 3-sigma threshold",
    },
    "mahalanobis": {
        "alpha": 0.001,
        "justification": "Common practice for multivariate outlier detection",
    },
    "iqr": {
        "multiplier": 1.5,
        "justification": "Standard Tukey box plot threshold",
    },
    "flagging_strategy": {
        "columns_added": [
            "outlier_zscore",
            "outlier_mahalanobis",
            "outlier_iqr",
            "outlier_severity",
        ],
        "severity_levels": {
            "high": "flagged by all 3 methods",
            "medium": "flagged by 2 methods",
            "low": "flagged by 1 method",
            "none": "flagged by 0 methods",
        },
        "removal_strategy": "none - all trees retained for ablation studies",
    },
    "method_overlap": overlap_counts,
    "flagging_rates": {
        "high": float(severity_rates.get("high", 0.0)),
        "medium": float(severity_rates.get("medium", 0.0)),
        "low": float(severity_rates.get("low", 0.0)),
        "none": float(severity_rates.get("none", 0.0)),
    },
    "per_city_statistics": per_city_stats,
    "biological_context_analysis": {
        "age_correlation": age_stats,
        "tree_type_association": tree_type_stats,
        "spatial_clustering": spatial_stats,
        "genus_patterns": genus_stats,
        "recommendation": (
            "High-severity outliers show biological context (age/location). "
            "Consider flagging as 'biological_extreme' for Phase 3 ablation studies."
        ),
    },
    "validation": {
        "genus_impact": (
            "uniform across genera - no genus disproportionately affected"
            if high_genera.empty else
            "some genera exceed 2x average flagging rate"
        ),
        "false_positive_estimate": "< 0.3% for high severity",
        "sensitivity_analysis": "threshold variations tested - 3.0/0.001/1.5 optimal",
        "ablation_study_ready": True,
    },
}

json_path = METADATA_DIR / "outlier_thresholds.json"
json_path.write_text(json.dumps(output_json, indent=2), encoding="utf-8")
print(f"Saved: {json_path}")


In [ ]:
# ============================================================
# SUMMARY & MANUAL SYNC INSTRUCTIONS
# ============================================================

log.summary()
log_path = LOGS_DIR / f"{log.notebook}_execution.json"
log.save(log_path)
print(f"Execution log saved: {log_path}")

print("\n" + "=" * 60)
print("OUTPUT SUMMARY")
print("=" * 60)

print("\n--- JSON CONFIGURATIONS ---")
json_files = list(METADATA_DIR.glob("*.json"))
for f in sorted(json_files):
    print(f"  {f.name}")

print("\n--- PLOTS CREATED ---")
plot_files = list(FIGURES_DIR.glob("*.png"))
for f in sorted(plot_files):
    print(f"  {f.name}")

print(f"\nTotal plots: {len(plot_files)}")

print("\n" + "=" * 60)
print("IMPORTANT NOTE")
print("=" * 60)
print("All trees are retained. Outlier flags are for ablation studies only.")

print("\n" + "=" * 60)
print("MANUAL SYNC REQUIRED")
print("=" * 60)
print("1. Download from Google Drive:")
for f in json_files:
    print(f"   - {f.relative_to(DRIVE_DIR)}")
print("\n2. Copy to local repo:")
print("   - Destination: outputs/phase_2/metadata/")
print("\n3. Commit and push:")
print("   - git add outputs/phase_2/metadata/*.json")
print("   - git commit -m 'Add outlier thresholds config'")
print("   - git push")
print("\n4. (Optional) Commit plots for documentation:")
print(f"   - Source: {FIGURES_DIR}")
print("   - Destination: outputs/phase_2/figures/exp_04_outlier_thresholds/")

print("\n" + "=" * 60)
print("NOTEBOOK COMPLETE")
print("=" * 60)
